In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the training and testing datasets
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/horse_health_outcomes/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/horse_health_outcomes/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Display the first few rows of the training dataset
print("Training Dataset Head:")
print(train_df.head())

# Display the first few rows of the testing dataset
print("\nTesting Dataset Head:")
print(test_df.head())

# Display the summary statistics of the training dataset
print("\nTraining Dataset Summary Statistics:")
print(train_df.describe())

# Display the summary statistics of the testing dataset
print("\nTesting Dataset Summary Statistics:")
print(test_df.describe())

# Display the data types of the training dataset
print("\nTraining Dataset Data Types:")
print(train_df.dtypes)

# Display the data types of the testing dataset
print("\nTesting Dataset Data Types:")
print(test_df.dtypes)

# Distinguish column types for tailored analysis and visualization
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
categorical_cols = train_df.select_dtypes(include=['object', 'category']).columns

print("\nNumeric Columns:")
print(numeric_cols)

print("\nCategorical Columns:")
print(categorical_cols)

# Visualize the correlation matrix for numeric columns
plt.figure(figsize=(12, 10))
correlation_matrix = train_df[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numeric Features')
plt.show()

# Visualize the distribution of the target variable
plt.figure(figsize=(8, 6))
sns.countplot(x=train_df['outcome'])
plt.title('Distribution of Horse Health Outcomes')
plt.show()

# Visualize the distribution of categorical features
for col in categorical_cols:
    plt.figure(figsize=(10, 6))
    sns.countplot(x=train_df[col])
    plt.title(f'Distribution of {col}')
    plt.xticks(rotation=45)
    plt.show()


Training Dataset Head:
  surgery  hospital_number  ...  capillary_refill_time     outcome
0     yes           527706  ...             less_3_sec        died
1     yes           528641  ...             less_3_sec       lived
2     yes           535043  ...             more_3_sec  euthanized
3     yes           535043  ...             less_3_sec  euthanized
4     yes           528890  ...             more_3_sec        died

[5 rows x 9 columns]

Testing Dataset Head:
  surgery  hospital_number  ...  capillary_refill_time     outcome
0      no           535381  ...             less_3_sec  euthanized
1     yes           535029  ...             less_3_sec  euthanized
2     yes           529461  ...             more_3_sec        died
3     yes           534157  ...             less_3_sec  euthanized
4     yes           529777  ...             less_3_sec       lived

[5 rows x 9 columns]

Training Dataset Summary Statistics:
       hospital_number  rectal_temp       pulse  respiratory_rate
co

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-09-15 07:45:35.873 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['surgery', 'peripheral_pulse', 'mucous_membrane', 'capillary_refill_time', 'outcome'], 'Numeric': ['hospital_number', 'rectal_temp', 'pulse', 'respiratory_rate'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Copy the DataFrames to avoid modifying the original data
train_df_processed = train_df.copy()
test_df_processed = test_df.copy()

# Handle missing values
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
categorical_cols = train_df.select_dtypes(include=['object', 'category']).columns

# Fill missing values for numeric columns with mean
fill_missing_numeric = FillMissingValue(features=numeric_cols, strategy='mean')
train_df_processed = fill_missing_numeric.fit_transform(train_df_processed)
test_df_processed = fill_missing_numeric.transform(test_df_processed)

# Fill missing values for categorical columns with most frequent value
fill_missing_categorical = FillMissingValue(features=categorical_cols, strategy='most_frequent')
train_df_processed = fill_missing_categorical.fit_transform(train_df_processed)
test_df_processed = fill_missing_categorical.transform(test_df_processed)

# Encode categorical variables using label encoding
label_encode = LabelEncode(features=categorical_cols)
train_df_processed = label_encode.fit_transform(train_df_processed)
test_df_processed = label_encode.transform(test_df_processed)

# Normalize numerical features
standard_scale = StandardScale(features=numeric_cols)
train_df_processed = standard_scale.fit_transform(train_df_processed)
test_df_processed = standard_scale.transform(test_df_processed)

# Display the processed data
print("Processed Training Dataset Head:")
print(train_df_processed.head())
print("\nProcessed Testing Dataset Head:")
print(test_df_processed.head())


Processed Training Dataset Head:
   surgery  hospital_number  ...  capillary_refill_time  outcome
0        2        -0.322836  ...                      0        0
1        2        -0.322161  ...                      0        2
2        2        -0.317536  ...                      1        1
3        2        -0.317536  ...                      0        1
4        2        -0.321981  ...                      1        0

[5 rows x 9 columns]

Processed Testing Dataset Head:
   surgery  hospital_number  ...  capillary_refill_time  outcome
0        0        -0.317292  ...                      0        1
1        2        -0.317546  ...                      0        1
2        2        -0.321568  ...                      1        0
3        2        -0.318176  ...                      0        1
4        2        -0.321340  ...                      0        2

[5 rows x 9 columns]


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df_processed)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['surgery', 'hospital_number', 'rectal_temp', 'pulse', 'respiratory_rate', 'peripheral_pulse', 'mucous_membrane', 'capillary_refill_time', 'outcome'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

# Assuming train_df_processed and test_df_processed are already defined from previous tasks
X_train = train_df_processed.drop('outcome', axis=1)
y_train = train_df_processed['outcome']

# Initialize and train the RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on the test set
X_test = test_df_processed.drop('outcome', axis=1)
y_test = test_df_processed['outcome']
y_pred = model.predict(X_test)

# Calculate F1 score
f1 = f1_score(y_test, y_pred, average='weighted')
print(f"F1 Score on Test Set: {f1}")

# Save the predictions to a CSV file
predictions_df = pd.DataFrame({'hospital_number': test_df['hospital_number'], 'predicted_outcome': y_pred})
predictions_df.to_csv('horse_health_predictions.csv', index=False)
print("Predictions saved to 'horse_health_predictions.csv'")


F1 Score on Test Set: 0.6703259660184822
Predictions saved to 'horse_health_predictions.csv'


In [6]:
import pandas as pd
from sklearn.metrics import f1_score

# Load the preprocessed test data
test_df_processed = pd.read_csv('processed_test.csv')

# Ensure the test data is preprocessed in the same way as the training data
numeric_cols = test_df.select_dtypes(include=[np.number]).columns
categorical_cols = test_df.select_dtypes(include=['object', 'category']).columns

# Apply the same transformations as in the training data preprocessing
fill_missing_numeric = FillMissingValue(features=numeric_cols, strategy='mean')
test_df_processed = fill_missing_numeric.transform(test_df_processed)

fill_missing_categorical = FillMissingValue(features=categorical_cols, strategy='most_frequent')
test_df_processed = fill_missing_categorical.transform(test_df_processed)

label_encode = LabelEncode(features=categorical_cols)
test_df_processed = label_encode.transform(test_df_processed)

standard_scale = StandardScale(features=numeric_cols)
test_df_processed = standard_scale.transform(test_df_processed)

# Separate features and target variable
X_test = test_df_processed.drop('outcome', axis=1)
y_test = test_df_processed['outcome']

# Load the trained model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Predict on the test set
y_pred = model.predict(X_test)

# Calculate the F1 score
f1 = f1_score(y_test, y_pred, average='weighted')
print(f"F1 Score on Test Set: {f1}")

# Save the predictions
predictions_df = pd.DataFrame({'hospital_number': test_df['hospital_number'], 'predicted_outcome': y_pred})
predictions_df.to_csv('horse_health_predictions.csv', index=False)
print("Predictions saved to 'horse_health_predictions.csv'")


FileNotFoundError: [Errno 2] No such file or directory: 'processed_test.csv'